# Lab: deterministic capacity and bottleneck simulation
No real load, network, sleeps, or flaky timing. Measurements are synthetic data produced by explicit formulas.


In [ ]:
from statistics import quantiles
print('Python environment ready')


## Objectives
Write a workload model, compare latency/throughput/concurrency/capacity, identify one bottleneck, apply one intervention, and test bounded/failure behavior.


## Baseline reproduction — Predict 1
If list latency grows with record count while CPU stays low, is an unbounded query a stronger bottleneck hypothesis than adding instances? Predict before running.


In [ ]:
baseline = {'records': [100, 1000, 10000], 'latency_ms': [8, 35, 180], 'cpu_pct': [20, 22, 25], 'throughput_rps': [40, 35, 20]}
assert baseline['latency_ms'][-1] > baseline['latency_ms'][0]
assert baseline['cpu_pct'][-1] < 30
print('Baseline evidence points to the list/query path')


**Pre-edit hypothesis:** enforcing a page limit will reduce worst-case latency by bounding records returned; it will not prove database scaling or improve every workload.


## Predict 2
If throughput is 20 requests/second and average latency is 0.5 seconds, is concurrency approximately 10? Yes; throughput × latency is a useful capacity intuition.


In [ ]:
throughput = 20; average_latency = 0.5
concurrency = throughput * average_latency
assert concurrency == 10
print('Approximate concurrency:', concurrency)


## Predict 3
If a cache key omits owner ID, can Ana receive Ben's cached note? Yes. Tenant identity belongs in the key, and invalidation/failure policy must be explicit.


In [ ]:
def cache_key(owner, note_id): return f'{owner}:{note_id}'
assert cache_key('ana', 7) != cache_key('ben', 7)
print('Tenant-safe key policy selected')


## Guided TODO: one intervention
Implement the expected bounded work in your notes before reading the reference. The next code cell is the executable reference solution; the fake model subtracts a fixed query cost when a page limit is enforced.


In [ ]:
def simulate(records, page_limit=None, dependency_ok=True):
    if page_limit is None:
        latency = 5 + records * 0.018
        payload = records
    else:
        latency = 5 + min(records, page_limit) * 0.018
        payload = min(records, page_limit)
    if not dependency_ok: return {'status': 503, 'latency_ms': latency, 'error': 'dependency-unavailable'}
    return {'status': 200, 'latency_ms': latency, 'payload_rows': payload}
before = simulate(10000)
after = simulate(10000, page_limit=100)
assert after['latency_ms'] < before['latency_ms'] and after['payload_rows'] == 100


In [ ]:
# Reference positive, negative, and failure checks.
assert simulate(0, page_limit=100)['payload_rows'] == 0
assert simulate(10000, page_limit=100)['status'] == 200
assert simulate(10000, page_limit=0)['payload_rows'] == 0  # policy would reject zero in a real API
assert simulate(10000, page_limit=100, dependency_ok=False)['status'] == 503
print({'before': before, 'after': after})


The zero-limit assertion is a prompt to add an API validation layer; the simulation itself returns an empty bounded page. In a production contract, reject page size below one and above a maximum.


## Intentionally weak AI-style design
A generated proposal might add three microservices, an unbounded retry, and a global cache before measuring. It creates new failure domains and may move the bottleneck. One measured intervention gives clearer evidence.


## Independent challenge — attempt before checking
Write assumptions for 10, 1,000, and 100,000 users and name the next bottleneck after pagination. Calculate the rates in notes first, then compare with the executable check below.


In [ ]:
workload = {10: 10*2/60, 1000: 1000*2/60, 100000: 100000*2/60}
assert workload[10] < workload[1000] < workload[100000]
decision = {'hypothesis_supported': after['latency_ms'] < before['latency_ms'], 'next_bottleneck':'database connections or dependency latency', 'unsafe_claim':'not a production user capacity guarantee'}
assert decision['hypothesis_supported'] and decision['unsafe_claim']
print(workload, decision)


## Exit questions
1. Which measurement supported the change?
2. What does it not prove?
3. Why one intervention?

### Answers
1. Latency and payload work grew with records and fell when page size was bounded.
2. Real p95/p99, CPU/memory, network, database indexes, and production capacity.
3. It preserves causal attribution.

## Evidence handoff
Save workload assumptions, before/after fake results, failure output, tenant-safe key, decision record, failure-domain sketch, and AI review.
